# 2. Requesting GPUs on NRP (Instructor: Daniel Diaz)

This session guides participants through the process of requesting GPUs and other compute resources on the National Research Platform (NRP). Attendees will be introduced to the NRP portal, where they can explore available hardware, including standard GPUs and the Qualcomm Cloud AI 100 Ultra SoCs. The tutorial demonstrates how to submit a resource request, select the appropriate GPU type for a given workload, and understand scheduling constraints and allocation quotas.

## GPUs on Kubernetes

Let's start by running a simple GPU job on Kubernetes. We will use the `nvidia-smi` command to check the GPUs available on the node.

*(The YAML configuration has been saved to the `yamls/` directory. If a YAML uses a fixed resource name with `<username>`, replace it with your username to avoid collisions.)*

In [ ]:
# Run this cell once to apply the NRP Tutorial theme (optional)
from IPython.display import HTML, display
display(HTML("""
<style>
  /* NRP Tutorial theme — indigo / slate / teal */
  .jp-MarkdownCell .jp-Cell-inputWrapper,
  .text_cell .input_area {
    background: linear-gradient(90deg, #eef2ff 0%, #f5f3ff 10%, #fafafa 22%);
    border-left: 4px solid #6366f1;
    padding: 0.75em 0 0.75em 1em;
    margin: 0.5em 0;
    border-radius: 0 8px 8px 0;
    box-shadow: 0 1px 3px rgba(99, 102, 241, 0.08);
  }
  .jp-MarkdownCell h1, .text_cell_render h1 {
    color: #312e81;
    font-weight: 700;
    border-bottom: 2px solid #6366f1;
    padding-bottom: 0.45em;
    margin-top: 0;
    background: linear-gradient(90deg, #e0e7ff 0%, rgba(224, 231, 255, 0.4) 70%, transparent 100%);
    padding: 0.35em 0 0.45em 0.6em;
    margin: 0 -0.6em 0.35em -0.6em;
    border-radius: 6px 6px 0 0;
  }
  .jp-MarkdownCell h2, .text_cell_render h2 {
    color: #4338ca;
    font-weight: 600;
    margin-top: 1.25em;
    margin-bottom: 0.45em;
    padding-left: 0.5em;
    border-left: 4px solid #6366f1;
    background: linear-gradient(90deg, rgba(99, 102, 241, 0.06) 0%, transparent 100%);
    padding: 0.25em 0.5em 0.25em 0.6em;
    border-radius: 0 4px 4px 0;
  }
  .jp-MarkdownCell h3, .text_cell_render h3 {
    color: #0f766e;
    font-weight: 600;
    margin-top: 1em;
    padding-left: 0.45em;
    border-left: 4px solid #14b8a6;
    background: linear-gradient(90deg, rgba(20, 184, 166, 0.06) 0%, transparent 100%);
    padding: 0.2em 0.45em 0.2em 0.55em;
    border-radius: 0 4px 4px 0;
  }
  .jp-MarkdownCell p, .jp-MarkdownCell li, .text_cell_render p, .text_cell_render li {
    line-height: 1.7;
    color: #374151;
  }
  .jp-MarkdownCell a, .text_cell_render a {
    color: #4f46e5;
    font-weight: 500;
  }
  .jp-MarkdownCell a:hover, .text_cell_render a:hover {
    color: #4338ca;
    text-decoration: underline;
  }
  .jp-MarkdownCell code, .text_cell_render code {
    background: #e0e7ff;
    color: #3730a3;
    padding: 0.22em 0.5em;
    border-radius: 5px;
    font-size: 0.9em;
    border: 1px solid #c7d2fe;
  }
  .jp-MarkdownCell pre, .text_cell_render pre {
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    border-left: 4px solid #6366f1;
    border-radius: 6px;
    padding: 0.8em 1em;
  }
  .jp-MarkdownCell strong, .text_cell_render strong {
    color: #1e293b;
  }
  .jp-MarkdownCell hr, .text_cell_render hr {
    border: none;
    border-top: 2px solid #cbd5e1;
    margin: 1.5em 0;
  }
  .jp-MarkdownCell ul, .text_cell_render ul {
    padding-left: 1.5em;
  }
  .jp-MarkdownCell li::marker, .text_cell_render li::marker {
    color: #6366f1;
  }
  .jp-MarkdownCell table, .text_cell_render table {
    border-collapse: collapse;
    border: 1px solid #e2e8f0;
    border-radius: 6px;
    overflow: hidden;
  }
  .jp-MarkdownCell th, .text_cell_render th {
    background: #eef2ff;
    color: #3730a3;
    padding: 0.5em 0.75em;
    text-align: left;
  }
  .jp-MarkdownCell td, .text_cell_render td {
    padding: 0.45em 0.75em;
    border-top: 1px solid #e2e8f0;
  }
</style>
"""))

### Cluster capacity and use policies

Before requesting GPUs, here's how the cluster is set up and what policies apply. Nautilus nodes offer a range of resources per node: **CPUs** (16–384 cores), **GPUs** (up to 16), **FPGAs** (up to 4), **memory** (16 GB–1.6 TB), and **local scratch** (107 GB–17.8 TB). To see what’s available across the cluster, check the [Resources](https://nrp.ai/viz/resources) page and the [Grafana cluster summary](https://grafana.nrp-nautilus.io/d/KQsJTWPiz/cluster-summary-nrp?orgId=1&from=now-1h&to=now&timezone=browser).

A few rules to keep in mind: interactive pods (no controller) are limited to **6 hours** and to 2 GPUs, 32 GB RAM, and 16 CPU cores. For batch work, use **Jobs** and avoid `sleep` in job specs. Always set both `requests` and `limits` for your resources. Full details: [Cluster policies](https://nrp.ai/documentation/userdocs/start/policies/) and [GPU pods](https://nrp.ai/documentation/userdocs/running/gpu-pods/).

## Kubernetes and Core Concepts

Before diving into deploying workloads, let's establish a baseline understanding of the core technologies that power the National Research Platform:

### Containers and Images
- **Containers**: Lightweight, standalone, executable packages that include everything needed to run a piece of software, including the code, runtime, system tools, and libraries.
- **Container Images**: The read-only templates used to create containers. In Nautilus, we use various images like the `ghcr.io/quic/cloud_ai_inference_ubuntu24` environment for Qualcomm workflows.

### Kubernetes Fundamentals
Kubernetes is an open-source container orchestration engine for automating deployment, scaling, and management of containerized applications.
- **Pods**: The smallest deployable computing units in Kubernetes. A pod can contain one or more tightly coupled containers. When you launch a JupyterHub instance, you are running inside a Pod.
- **Jobs**: Workloads designed to reliably execute a task to completion (e.g., training a model once), as opposed to long-running services.
- **Persistent Volume Claims (PVCs)**: Kubernetes resources that request storage allocation. They ensure your data persists independently of the lifecycle of any single pod.

### Hardware Acceleration
Kubernetes requires specialized extensions to manage and assign non-CPU hardware.
- **GPUs in Kubernetes**: Workloads can explicitly request NVIDIA GPUs (e.g. `nvidia.com/gpu`) by specifying resource limits in their deployment manifests.
- **Device Plugins**: Software components that advertise specialized hardware resources to the Kubernetes scheduler. For example, Qualcomm Cloud AI devices are exposed through a device plugin natively mapping as `qualcomm.com/qaic`.

### Basic kubectl Usage
The Kubernetes command-line tool, `kubectl`, allows you to run commands against Kubernetes clusters. You will use commands like `kubectl apply -f <file.yaml>` to deploy resources, or `kubectl get pods` to inspect running states.


## GPU Provisioning on NRP

To run workloads on GPUs or Qualcomm AI100 devices, you must **request the right resources and use compatible images**. Below is a concise reference; see the [NRP GPU types guide](https://nrp.ai/documentation/userdocs/running/gpu-pods/#choosing-gpu-type) for more detail.

### NVIDIA GPUs

- **Resource request:** In your pod spec, set `resources.limits` and `resources.requests` with the GPU resource key. For a generic GPU use `nvidia.com/gpu: <count>`. For a specific product (e.g. A100, A10, L40, RTX A6000, V100) use the product-specific resource (e.g. `nvidia.com/a100`, `nvidia.com/rtxa6000`) if your cluster exposes them.
- **Node selector (optional):** To target a specific GPU type, add a `nodeSelector` or `affinity` that matches node labels. For example, NRP nodes may have `nvidia.com/gpu.product` (e.g. `NVIDIA-A10`, `NVIDIA-L40`, `Tesla-V100-SXM2-32GB`). Use this when you need a particular GPU model.
- **Runtime / image:** Use a CUDA-enabled image (e.g. `nvidia/cuda:12.0.0-base-ubuntu22.04` or NRP scientific images with CUDA). The node runs the NVIDIA device plugin and container runtime so the GPU is available inside the container.
- **Example (generic GPU):** See `yamls/gpu-pod.yaml` — it requests `nvidia.com/gpu: 1` and runs `nvidia-smi`. Replace `<username>` in the pod name if sharing the namespace.

### Qualcomm Cloud AI 100

- **Resource request:** Use `qualcomm.com/qaic: <count>` in `resources.limits` and `resources.requests`. Each unit corresponds to one SoC. Nautilus has **8 Qualcomm Cloud AI 100 Ultra cards**, each with **4 SoCs**, for **32 devices** in total; each device can run LLMs up to roughly 25B parameters.
- **Node selector:** Nodes with Qualcomm devices are labeled; use `nodeAffinity` with `qualcomm.com/qaic: Exists` (or the label your cluster uses) so the pod is scheduled on a node that has the device plugin and hardware.
- **Runtime / image:** Use a Qualcomm-supported image. For vLLM-based inference use the NRP image `gitlab-registry.nrp-nautilus.io/cloud-ai-100/qaic-docker-images:vllm-latest`; for SDK examples see `ghcr.io/quic/cloud_ai_inference_ubuntu24` (Part 3 and Part 4). The container must have the Cloud AI SDK and drivers expected by the device plugin.
- **Example:** See `yamls/qaic-inference.yaml` in Part 3/4 for a full inference pod using `qualcomm.com/qaic: 1`. Further details: [Qualcomm Cloud AI 100](https://nrp.ai/documentation/userdocs/qaic/) and the [Cloud AI SDK vLLM guide](https://quic.github.io/cloud-ai-sdk-pages/latest/Getting-Started/Installation/vLLM/vLLM/).

Create the GPU pod with the command below. **Replace `<username>` in the pod name in `yamls/gpu-pod.yaml`** with your username first.

The GPU pod YAML is in `yamls/gpu-pod.yaml`. It requests one NVIDIA GPU and runs `nvidia-smi`:

```bash
kubectl apply -n nrp-training-k8s -f yamls/gpu-pod.yaml
```

In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/gpu-pod.yaml

## MPI On Kubernetes

First lets run a basic CPU MPI job on Kubernetes. We will use the MPI operator to run the job.
We will calculate the value of Pi using the Monte Carlo method. The code is written in C and is available in the `mpi-pi` directory.

*(The YAML configuration has been saved to the `yamls/` directory)*

Use the YAML from `yamls/mpi-pi.yaml` and create the job:

```bash
kubectl apply -n nrp-training-k8s -f yamls/mpi-pi.yaml
```

In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/mpi-pi.yaml

Now let's run a multi-node GPU MPI job on Kubernetes. We will use the MPI operator to run the job.

*(The YAML configuration has been saved to the `yamls/` directory)*

Use the YAML from `yamls/mpi-tensorflow.yaml` and create the job:

```bash
kubectl apply -n nrp-training-k8s -f yamls/mpi-tensorflow.yaml
```

In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/mpi-tensorflow.yaml

## End

**Please make sure you did not leave any running pods. Jobs and associated completed pods are OK.**

**Join NRP & hands-on support:** Use these links to [get started with NRP](https://nrp.ai/documentation/userdocs/start/getting-started/) and to [join NRP's Matrix chat](https://nrp.ai/contact/) for hands-on support during the tutorial.

**Related documentation:** [Using Nautilus](https://nrp.ai/documentation/userdocs/start/using-nautilus/) · [GPU pods](https://nrp.ai/documentation/userdocs/running/gpu-pods/) · [Cluster policies](https://nrp.ai/documentation/userdocs/start/policies/) · [Qualcomm Cloud AI 100](https://nrp.ai/documentation/userdocs/qaic/)